# SMARTCARE HOSPITAL AI COURSEWORK
## Task 05 — Machine Learning Model Development

This notebook develops and compares four classification models. Model selection is based on training-only stratified cross-validation, while the validation set provides a supporting performance check. The test set remains untouched for final evaluation in Task 06.

## 1. Import Required Libraries

In [3]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
Path('models').mkdir(exist_ok=True)

## 2. Load the Task 03 Output

The unscaled processed dataset is used because scaling and feature selection are fitted inside each training pipeline. Sensitive attributes are retained separately for later fairness evaluation and are not used as model inputs.

In [4]:
df_model = pd.read_csv('smartcare_preprocessed_unscaled.csv')

assert df_model.shape == (875, 26), 'Run the corrected Task 03 notebook first.'
assert 'no_show' in df_model.columns
assert 'long_wait_flag' in df_model.columns
assert df_model.isnull().sum().sum() == 0

raw_data = pd.read_csv('smartcare_ai_dataset_1000.csv')
confirmed = raw_data[raw_data['appointment_status'].isin(['Completed', 'No-Show'])].reset_index(drop=True)
assert len(confirmed) == len(df_model)

fairness_cols = confirmed[['gender', 'age']].copy()
fairness_cols['age_group'] = pd.cut(
    fairness_cols['age'], bins=[0, 17, 39, 59, np.inf],
    labels=['0-17', '18-39', '40-59', '60+']
)

print('Model dataset:', df_model.shape)
print('Candidate inputs:', df_model.shape[1] - 1)
display(df_model.head(3))

Model dataset: (875, 26)
Candidate inputs: 25


,age,waiting_days,previous_appointments,missed_previous_appointments,previous_admissions,no_show,appointment_month,appointment_dayofweek,department_General Medicine,department_Laboratory Services,department_Neurology,department_Orthopedics,department_Pediatrics,department_Radiology,diagnosis_Back Pain,diagnosis_Chest Pain,diagnosis_Diabetes,diagnosis_Fever,diagnosis_Fracture,diagnosis_Hypertension,diagnosis_Kidney Infection,diagnosis_Migraine,diagnosis_Pneumonia,missed_ratio,has_missed_before,long_wait_flag
0,53,10,1,0,1,0,4,3,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.000,0,0
1,26,2,3,1,0,0,5,3,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.250,1,0
2,22,22,7,1,1,1,7,2,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0.125,1,1


## 3. Train, Validation and Test Split

A stratified 60/20/20 split preserves the class distribution. Cross-validation is performed only within the training set. The validation set is used for a supporting check, and the test set is saved without evaluation for Task 06.

In [5]:
TARGET = 'no_show'
X = df_model.drop(columns=TARGET)
y = df_model[TARGET]

X_temp, X_test, y_temp, y_test, fair_temp, fair_test = train_test_split(
    X, y, fairness_cols, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val, fair_train, fair_val = train_test_split(
    X_temp, y_temp, fair_temp, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_temp
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Train:', len(X_train), 'Validation:', len(X_val), 'Test:', len(X_test))
print('Train class balance:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Validation class balance:', y_val.value_counts(normalize=True).round(3).to_dict())
print('Test class balance:', y_test.value_counts(normalize=True).round(3).to_dict())

Train: 525 Validation: 175 Test: 175
Train class balance: {1: 0.566, 0: 0.434}
Validation class balance: {1: 0.566, 0: 0.434}
Test class balance: {1: 0.566, 0: 0.434}


## 4. Training-Only Ablation Study for `long_wait_flag`

The same Random Forest pipeline is compared with and without `long_wait_flag`. Only the training set is used, so the validation and test sets remain independent. F1-score, recall and ROC-AUC are compared because the target is moderately imbalanced.

In [6]:
ablation_scoring = {'F1 Score': 'f1', 'Recall': 'recall', 'ROC-AUC': 'roc_auc'}

def run_ablation(features, label):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE
        ))
    ])
    scores = cross_validate(
        pipeline, X_train[features], y_train,
        cv=cv_strategy, scoring=ablation_scoring, n_jobs=-1
    )
    return {
        'Version': label,
        'Features': len(features),
        'Mean F1': scores['test_F1 Score'].mean(),
        'Mean Recall': scores['test_Recall'].mean(),
        'Mean ROC-AUC': scores['test_ROC-AUC'].mean()
    }

features_with_flag = list(X_train.columns)
features_without_flag = [c for c in X_train.columns if c != 'long_wait_flag']

ablation_df = pd.DataFrame([
    run_ablation(features_without_flag, 'Without long_wait_flag'),
    run_ablation(features_with_flag, 'With long_wait_flag')
])
display(ablation_df.round(4))

metric_columns = ['Mean F1', 'Mean Recall', 'Mean ROC-AUC']
ablation_change = ablation_df.loc[1, metric_columns] - ablation_df.loc[0, metric_columns]
display(ablation_change.to_frame('Change with flag').T.round(4))

if ablation_change['Mean F1'] <= 0:
    X_train = X_train.drop(columns='long_wait_flag')
    X_val = X_val.drop(columns='long_wait_flag')
    X_test = X_test.drop(columns='long_wait_flag')
    print('Decision: long_wait_flag excluded; continuous waiting_days retained.')
else:
    print('Decision: long_wait_flag retained because mean CV F1 improved.')

,Version,Features,Mean F1,Mean Recall,Mean ROC-AUC
0,Without long_wait_flag,24,0.6567,0.7103,0.5926
1,With long_wait_flag,25,0.6515,0.7070,0.5808


,Mean F1,Mean Recall,Mean ROC-AUC
Change with flag,-0.005151,-0.003333,-0.011868


Decision: long_wait_flag excluded; continuous waiting_days retained.


**Interpretation:** Mean cross-validation F1 decreased slightly when `long_wait_flag` was added. Recall and ROC-AUC also decreased. Therefore, the flag is excluded from the final model inputs, while continuous `waiting_days` is retained. The flag remains documented in Task 03 as a candidate engineered feature, but the final modelling decision is based only on training-set cross-validation.

## 5. Define the Model Pipelines

Each pipeline fits scaling and feature selection separately within every cross-validation fold. After the ablation decision, `SelectKBest` retains 20 of the remaining 24 candidate features, removing only the weakest individual signals. Balanced class weights are used where supported; Gradient Boosting receives balanced sample weights during fitting.

In [7]:
def make_pipeline(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=20)),
        ('model', model)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE)
    ),
    'Gradient Boosting': make_pipeline(
        GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)
    ),
    'SVM (RBF)': make_pipeline(
        SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE)
    )
}

scoring = {
    'Accuracy': 'accuracy', 'Precision': 'precision', 'Recall': 'recall',
    'F1 Score': 'f1', 'ROC-AUC': 'roc_auc'
}
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)
print('Four model pipelines prepared.')

Four model pipelines prepared.


## 6. Stratified Cross-Validation and Model Selection

Models are ranked by mean cross-validation F1-score. F1 is the primary metric because low recall misses high-risk appointments, while low precision creates unnecessary reminder calls. Accuracy and ROC-AUC are reported as supporting measures.

In [8]:
cv_rows = []

for name, pipeline in models.items():
    fit_params = {'model__sample_weight': sample_weights_train} if name == 'Gradient Boosting' else None
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv_strategy,
        scoring=scoring, n_jobs=-1, params=fit_params
    )
    cv_rows.append({
        'Model': name,
        **{f'CV Mean {metric}': scores[f'test_{metric}'].mean() for metric in scoring},
        'CV F1 Std': scores['test_F1 Score'].std()
    })

cv_df = (pd.DataFrame(cv_rows)
         .sort_values('CV Mean F1 Score', ascending=False)
         .reset_index(drop=True))
display(cv_df.round(4))

selected_model_name = cv_df.loc[0, 'Model']
print('Selected model by training-only CV F1:', selected_model_name)

,Model,CV Mean Accuracy,CV Mean Precision,CV Mean Recall,CV Mean F1 Score,CV Mean ROC-AUC,CV F1 Std
0,Random Forest,0.5771,0.6085,0.7036,0.6519,0.5876,0.0399
1,Logistic Regression,0.5981,0.6561,0.6229,0.6370,0.6273,0.0111
2,SVM (RBF),0.5714,0.6317,0.5856,0.6070,0.6037,0.0293
3,Gradient Boosting,0.5505,0.6004,0.6094,0.6041,0.5462,0.0350


Selected model by training-only CV F1: Random Forest


## 7. Fit Models and Check Validation Performance

After model selection is fixed by cross-validation, each pipeline is fitted on the complete training set. Validation results provide a transparent supporting check but do not change the selected model.

In [9]:
fitted_models = {}
validation_rows = []

for name, pipeline in models.items():
    if name == 'Gradient Boosting':
        pipeline.fit(X_train, y_train, model__sample_weight=sample_weights_train)
    else:
        pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

    predictions = pipeline.predict(X_val)
    probabilities = pipeline.predict_proba(X_val)[:, 1]
    validation_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, predictions),
        'Precision': precision_score(y_val, predictions),
        'Recall': recall_score(y_val, predictions),
        'F1 Score': f1_score(y_val, predictions),
        'ROC-AUC': roc_auc_score(y_val, probabilities)
    })

validation_df = pd.DataFrame(validation_rows).sort_values('F1 Score', ascending=False)
display(validation_df.round(4))
print('Final selected model remains:', selected_model_name)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
1,Random Forest,0.6400,0.6475,0.7980,0.7149,0.6310
0,Logistic Regression,0.6057,0.6562,0.6364,0.6462,0.6628
2,Gradient Boosting,0.5886,0.6337,0.6465,0.6400,0.6478
3,SVM (RBF),0.5600,0.6196,0.5758,0.5969,0.6235


Final selected model remains: Random Forest


## 8. Save the Models and Data Splits

The complete pipelines are saved, so scaling and feature selection are applied consistently in later tasks. The untouched test set is stored for one-time final evaluation in Task 06.

In [11]:
from pathlib import Path

Path('models').mkdir(exist_ok=True)

# Save all models for Task 06 comparison
for name, pipeline in fitted_models.items():
    filename = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    joblib.dump(pipeline, f'models/{filename}_pipeline.pkl')

# Save the selected model
joblib.dump(
    fitted_models[selected_model_name],
    'models/selected_model_pipeline.pkl'
)

# Save data needed for Task 06 and Task 07
X_train.to_csv('models/X_train.csv', index=False)
X_val.to_csv('models/X_val.csv', index=False)
y_val.to_csv('models/y_val.csv', index=False)

X_test.to_csv('models/X_test.csv', index=False)
y_test.to_csv('models/y_test.csv', index=False)
fair_test.to_csv('models/fair_test.csv', index=False)

# Save Task 05 result tables
ablation_df.to_csv(
    'models/long_wait_flag_ablation.csv',
    index=False
)

cv_df.to_csv(
    'models/cross_validation_comparison.csv',
    index=False
)

# Save model-selection information
selection_info = {
    'selected_model': selected_model_name,
    'selection_metric': (
        'Mean F1-score from training-only '
        '5-fold stratified cross-validation'
    ),
    'feature_columns': list(X_train.columns),
    'feature_selection_k': 20,
    'random_state': RANDOM_STATE,
    'train_size': len(X_train),
    'validation_size': len(X_val),
    'test_size': len(X_test)
}

with open('models/selected_model_info.json', 'w') as file:
    json.dump(selection_info, file, indent=2)

print('Required Task 05 files saved successfully.')

Required Task 05 files saved successfully.


## Task 05 Summary

- Four classification pipelines were developed and compared.
- Scaling and feature selection were learned inside each training fold.
- `long_wait_flag` was evaluated using training-only cross-validation and excluded because it did not improve mean F1-score.
- The model was selected using mean cross-validation F1-score.
- Validation was used only as a supporting check.
- The test set was not evaluated and remains independent for Task 06.